# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I frame the Content Refresh problem as a classification task. The model will classify content items into whether they should be prioritized for a refresh based on observed search-performance signals. Classification fits because the final decision is a discrete action: prioritize a content item for refresh or do not prioritize it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


Rows: 27934
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target will be a defined proxy rather than a directly observed business outcome. I will define a refresh-priority label using observed search-performance signals, such as impressions, clicks, search volume, and ranking position. A content item will be marked as high priority when its observed performance suggests meaningful opportunity or decline. This label is a decision-support proxy and does not prove that refreshing the content will improve performance.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if "impressions_90d" in df.columns:
    threshold = df["impressions_90d"].median()
    df["refresh_priority"] = (
        df["impressions_90d"] < threshold
    ).astype(int)

print(df["refresh_priority"].value_counts())

refresh_priority
0    13968
1    13966
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use F1-score as the primary success metric because the task is classification and both false positives and false negatives matter. A high F1-score means the model can identify refresh-priority content while avoiding too many incorrect recommendations. As a supporting check, I will also examine precision and recall.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import f1_score, precision_score, recall_score

print("Metric: F1-score")

Metric: F1-score


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content item. Each row represents one anonymized piece of content and contains search-performance signals that can be used to assess whether that content should be considered for a refresh.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nOne row represents one content item.")


Shape: (27934, 45)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,refresh_priority
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0



One row represents one content item.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
The pattern is too complex for a single fixed if-statement because content performance can depend on multiple signals at the same time. Search volume, impressions, clicks, and ranking position can interact differently across content items. ML can learn combinations of these observed signals and provide a consistent ranking or classification for decision-support. However, the model will not prove causality or predict Google's ranking decisions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
candidate_features = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "position_90d"
]

available_features = [
    col for col in candidate_features
    if col in df.columns
]

print("Available ML features:")
print(available_features)

print("\nFeature summary:")
display(df[available_features].describe())

Available ML features:
['search_volume', 'impressions_90d', 'clicks_90d']

Feature summary:


,search_volume,impressions_90d,clicks_90d
count,25636.000000,27933.000000,27933.000000
mean,158.067171,5197.421186,16.138832
std,1507.427270,16719.242107,76.150365
min,0.000000,1.000000,0.000000
25%,0.000000,82.000000,0.000000
50%,10.000000,732.000000,1.000000
75%,20.000000,3644.000000,7.000000
max,74000.000000,517715.000000,4178.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.